# Weekly Workflow — BBO Capstone

Slim notebook for each week: load data, fit surrogates, generate queries, format for portal.
Run cells in order. After receiving results, use **data_management.ipynb** to load and update, then return here for the next week.

**Week 13 (final query round):** shrink-and-defend the ceiling. W12 delivered two new bests (F2 = 0.706, F7 = 3.233) and five small regressions on near-converged functions. W13 therefore tightens every exploitation radius around the *current* best-observed point and only takes one genuine exploration step (F7, a shortened PC1 re-extrapolation).

Priorities per function:

- **F7** — still on a trajectory (2.97 → 3.11 → 3.23). GP-ARD + UCB β=1.4, focus = W12 best, radius 0.020. Only real bet of the round.
- **F2** — just found a new best at `[0.710, 0.940]`. GP + EI ξ=1e−4, radius 0.005 around the new best.
- **F3, F4, F6, F8** — at ceiling; return to the historical best-observed point with a halved radius. F4 freezes dims 2 and 3 via a per-axis radius (new helper capability) since every perturbation along those dims has regressed.
- **F5** — `[1,1,1,1]` is the confirmed max; submit a novel near-corner (deterministic oracle blocks exact duplicates).
- **F1** — 22 flat samples; max-min distance probe is as good as any other choice.

Best points are derived programmatically from the loaded history (no external JSON), so re-running after each weekly update auto-refreshes the focus centres.


## 1. Setup and load data

**Week 13 approach** — shrink radii around the *current* best-observed point per function; do not re-derive focus centres from PCA. Reasoning: W12 validated the PCA directions (F2 and F7 both delivered new bests in the proposed focus region) and the W12 observations themselves are the freshest data point on each function's peak.

Per-function W13 play:

- **F1**: max-min distance probe at `[0.333, 0.333]` (no signal after 22 samples).
- **F2**: GP + EI, tight around W12 new best `[0.710, 0.940]`, `r=0.005`.
- **F3**: GP-ARD + EI, tight around W5 best `[0.405, 0.392, 0.483]`, `r=0.003` (W12's 0.008 overshot).
- **F4**: GP-ARD + EI, W9 best-cluster centre, **per-axis radius `[0.002, 0.002, 0, 0]`** — dims 2 and 3 frozen; every perturbation on those has regressed since W8.
- **F5**: Manual near-corner `[0.99999, 0.99999, 0.99999, 0.99999]` (true max `[1,1,1,1]=8662` already banked; oracle blocks exact duplicates).
- **F6**: GP-ARD + EI, W8 best `[0.243, 0.300, 0.445, 0.775, 0.040]`, `r=0.008` (half of W12's 0.015 after a −0.5 % regression).
- **F7**: GP-ARD + UCB (β=1.4), focus = W12 best `[0.196, 0.144, 0.495, 0.246, 0.331, 0.698]`, `r=0.020`. β and radius both smaller than W12 since the trajectory is likely curving.
- **F8**: SVR + EI, tight around W10 best, `r=0.004` (half of W12's 0.008).

`initialize_all_weeks` auto-globs `data/results/week_*`, so the W12 results already form part of the history.


In [ ]:
import sys
from pathlib import Path
# Allow imports from src when run from notebooks/ or from project root
root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(root))

import numpy as np
from src.data import FunctionData, DATA_DIR, initialize_all_weeks
from src.surrogates import GPSurrogate, SVMSurrogate, MLPSurrogate
from src.acquisition import optimize_acquisition_enhanced, optimize_acquisition_with_regional_focus
from src.utils import format_for_portal, plot_progress, display_competition_summary

np.random.seed(42)
print("✓ Imports ready")
# If you get ImportError for SVMSurrogate/MLPSurrogate: use Kernel → Restart, then run again (clears cached bytecode).

In [ ]:
# Load all 8 functions (from data/function_1 .. function_8)
functions = {i: FunctionData(i, data_dir=DATA_DIR) for i in range(1, 9)}
for i, f in functions.items():
    print(f"  {f}")
print("✓ Functions loaded")

In [ ]:
# Initialize with all historical weeks (from data/results/week_1, week_2, ...)
num_weeks = initialize_all_weeks(functions)
print(f"✓ Initialized with {num_weeks} weeks of results")

## 2. Week 13 strategies (shrink-and-defend)

1. Focus centres are read directly from the loaded history — each function's best-observed input is derived from `func_data.get_best()`, so the dict auto-refreshes after every weekly update.
2. F4 uses a per-axis radius (new capability in `optimize_acquisition_with_regional_focus`) to freeze dims 2 and 3.
3. F7 is the only function using UCB and a non-trivial radius; everything else is a tight EI exploitation around the existing best.


### Why this structure for Week 13

- The deterministic oracle means **re-querying an exact duplicate is wasted**, so every focus centre is nudged (F5) or accompanied by a non-zero radius (everything else).
- The PCA work from W12 stays relevant — it told us which dims are active on each function. Here we translate that into **radii**: small on active dims, zero on frozen ones (F4 only, since it's the one place where dim-freezing changes behaviour).
- Grading is on the single best observed per function, so the W13 downside is strictly bounded — a bad query does not erase the historical best. That means we can afford the single exploration step on F7 without hedging it.


In [ ]:
# Week 13: derive focus centres programmatically from the loaded history.
# best_x[fid] is the input vector with the highest observed output across all weeks.
best_x = {}
best_y = {}
for fid, fd in functions.items():
    x_best, y_best = fd.get_best()
    best_x[fid] = np.asarray(x_best, dtype=float)
    best_y[fid] = float(y_best)

print("Current best-observed per function (focus centres for W13):")
for fid in range(1, 9):
    print(f"  F{fid}: y={best_y[fid]:+.6g}  x={np.array2string(best_x[fid], precision=4, separator=', ')}")


In [ ]:
# Week 13 strategies — shrink-and-defend (final query round, then grading).
CURRENT_WEEK = 13

# F1: max-min probe far from the 22 flat samples already taken.
f1_manual = np.array([0.333333, 0.333333])

# F5: [0.999999,...] was queried twice already (W6, W10). Nudge to a near-corner
# point that is new under the 6-dp portal format. Expected return still ~8662.
f5_manual = np.array([0.999998, 0.999998, 0.999998, 0.999998])

# F4: per-axis radius freezes dims 2 and 3 (every perturbation there since W8 has regressed).
f4_radius = np.array([0.002, 0.002, 0.0, 0.0])

strategies = {
    1: {"manual_query": f1_manual},
    2: {"surrogate": "gp", "gp_noise": 0.05, "acq_func": "ei", "xi": 1e-4,
        "use_regional_focus": True, "focus_region": best_x[2],
        "focus_radius": 0.005, "n_random": 2000},
    3: {"surrogate": "gp", "use_ard": True, "acq_func": "ei", "xi": 1e-4,
        "use_regional_focus": True, "focus_region": best_x[3],
        "focus_radius": 0.003, "n_random": 2000},
    4: {"surrogate": "gp", "use_ard": True, "acq_func": "ei", "xi": 5e-5,
        "use_regional_focus": True, "focus_region": best_x[4],
        "focus_radius": f4_radius, "n_random": 3000},
    5: {"manual_query": f5_manual},
    6: {"surrogate": "gp", "use_ard": True, "acq_func": "ei", "xi": 5e-4,
        "use_regional_focus": True, "focus_region": best_x[6],
        "focus_radius": 0.008, "n_random": 2000},
    7: {"surrogate": "gp", "use_ard": True, "acq_func": "ucb", "beta": 1.4,
        "use_regional_focus": True, "focus_region": best_x[7],
        "focus_radius": 0.020, "n_random": 20000},
    8: {"surrogate": "svr", "svr_C": 5.0, "svr_epsilon": 0.15, "svr_n_bootstrap": 40,
        "acq_func": "ei", "xi": 3e-4,
        "use_regional_focus": True, "focus_region": best_x[8],
        "focus_radius": 0.004, "n_random": 20000},
}

print(f"Strategies set for Week {CURRENT_WEEK}.")
print(f"  F1 max-min probe  : {np.array2string(f1_manual, precision=6)}")
print(f"  F2 focus (W12 new): {np.array2string(best_x[2], precision=6)}  r=0.005")
print(f"  F4 per-axis radius: {np.array2string(f4_radius, precision=6)}  (dims 2,3 frozen)")
print(f"  F5 near-corner    : {np.array2string(f5_manual, precision=6)}")
print(f"  F7 focus (W12 new): {np.array2string(best_x[7], precision=6)}  UCB β=1.4 r=0.020")


## 3. Generate queries

In [ ]:
def _make_surrogate(surrogate_key: str, func_data, strategy: dict):
    key = surrogate_key.lower()
    if key == "gp":
        use_ard = strategy.get("use_ard", False)
        noise = strategy.get("gp_noise", 1e-5)
        return GPSurrogate(length_scale=0.5, optimize=True, noise=noise, use_ard=use_ard)
    if key == "svr":
        return SVMSurrogate(
            C=strategy.get("svr_C", 10.0),
            epsilon=strategy.get("svr_epsilon", 0.1),
            n_bootstrap=strategy.get("svr_n_bootstrap", 20),
        )
    if key == "mlp":
        return MLPSurrogate(hidden_sizes=(64, 32), dropout=0.1, n_mc_samples=50)
    return GPSurrogate(length_scale=0.5, optimize=True)

queries = {}
surrogates = {}

for func_id in range(1, 9):
    func_data = functions[func_id]
    s = strategies.get(func_id, {"acq_func": "ucb", "beta": 2.0})
    if "manual_query" in s:
        x = np.asarray(s["manual_query"], dtype=float)
        queries[func_id] = x
        surrogates[func_id] = None
        print(f"  F{func_id}: MANUAL → x = {np.array2string(x, precision=4, separator=', ')}")
        continue
    surrogate_key = s.get("surrogate", "gp")
    surrogate = _make_surrogate(surrogate_key, func_data, s)
    surrogate.fit(func_data.inputs, func_data.outputs)
    acq_func = s.get("acq_func", "ucb")
    acq_params = {k: v for k, v in s.items() if k in ("beta", "xi")}
    if s.get("use_regional_focus") and s.get("focus_region") is not None:
        x, mu, sigma = optimize_acquisition_with_regional_focus(
            surrogate, func_data, acq_func=acq_func,
            focus_region=s["focus_region"], focus_radius=s.get("focus_radius", 0.15),
            n_random=s.get("n_random", 1000), expand_search=s.get("expand_search", True),
            random_state=42,
            **acq_params
        )
    else:
        x, mu, sigma = optimize_acquisition_enhanced(
            surrogate, func_data, acq_func=acq_func,
            n_random=s.get("n_random", 1000), **acq_params
        )
    queries[func_id] = x
    surrogates[func_id] = surrogate
    print(f"  F{func_id}: {acq_func.upper()} → x = {np.array2string(x, precision=4, separator=', ')}")

print("✓ Queries generated")

# Duplicate check against the 6-dp portal format — the oracle is deterministic,
# so any duplicate (at 6-dp precision) would be a wasted round.
print("\nDuplicate check (6-dp portal precision):")
any_dup = False
for func_id in range(1, 9):
    q_fmt = tuple(np.round(queries[func_id], 6))
    hist_fmt = {tuple(np.round(row, 6)) for row in functions[func_id].inputs}
    is_dup = q_fmt in hist_fmt
    any_dup = any_dup or is_dup
    print(f"  F{func_id}: {'DUPLICATE' if is_dup else 'new'}  →  {q_fmt}")
if any_dup:
    print("\n⚠  At least one query is a duplicate. Adjust the strategy or manual_query.")
else:
    print("\n✓ All 8 queries are non-duplicate at 6-dp precision.")

## 4. Format and submit

In [ ]:
format_for_portal(queries, title=f"WEEK {CURRENT_WEEK} QUERIES — READY FOR SUBMISSION")

## 5. Optional: view progress

In [ ]:
display_competition_summary(functions)
plot_progress(functions)